# Recommendation Engine

## Objective

The objective of this notebook is to build a recommendation engine using the engineered customer features and customer segmentation results generated in previous stages of the machine learning pipeline.

Unlike previous notebooks that directly accessed MySQL, this notebook consumes the feature datasets produced by Feature Engineering and Customer Segmentation.

The recommendation engine will:

- Load engineered customer features.
- Load customer segment assignments.
- Generate recommendation candidates.
- Build popularity-based recommendations.
- Build collaborative filtering recommendations.
- Produce Top-N personalized recommendations.

The generated recommendation artifacts will later be used by the Personalization Engine and exposed through FastAPI APIs.

In [ ]:
import os 
print(os.getcwd())

In [ ]:
import os
import sys

# Find the project root notebooks/
project_root = os.path.abspath("..")

# Add it to Python's import path if not already present
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

In [ ]:
## Import Required Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

import joblib

from src.database import get_engine

pd.set_option("display.max_columns", None)

## Load Engineered Datasets

Load the customer feature dataset and customer segmentation dataset generated in previous notebooks.

These datasets serve as the primary inputs for the recommendation engine.

In [ ]:
#go .. one dir up = root then go in data dir and then features dir
user_item_df = pd.read_csv(os.path.join("..","data","features",
    "user_item_features.csv")
)

print(f"Dataset Shape : {user_item_df.shape}")



In [ ]:
print("=" * 50)
print("user_item_df")
print("=" * 50)

print(user_item_df.shape)

user_item_df.head()

In [ ]:
user_item_df.info()

In [ ]:
user_item_df.isnull().sum()

In [ ]:
user_item_df.describe()

In [ ]:
user_item_df.duplicated().sum()

### Observation

The user-item interaction dataset is clean and ready for recommendation modeling.

Each record represents a unique interaction between a customer and a product, with interaction strength serving as an implicit feedback score.

# Phase 1 — Popularity-Based Recommendation

Popularity-Based Recommendation is the simplest recommendation strategy.

Instead of generating personalized recommendations, it recommends products that have received the highest overall interaction from all users.

This approach serves as a baseline model and provides recommendations even for new users with no interaction history (cold-start users).

## Compute Product Popularity

Aggregate interaction strength for each product.

Products with higher cumulative interaction strength are considered more popular.

In [ ]:
popular_items = (
    user_item_df
    .groupby("itemid")
    .agg(
        total_interaction_strength=(
            "interaction_strength",
            "sum"
        ),
        total_users=(
            "visitorid",
            "nunique"
        )
    )
    .reset_index()
)

In [ ]:
popular_items = (
    popular_items
    .sort_values(
        by="total_interaction_strength",
        ascending=False
    )
)

popular_items.head(10)

## Visualize the Most Popular Products

Visualizing the top products helps validate the popularity-based recommendation strategy and provides insight into customer engagement patterns.

In [ ]:
top_items = popular_items.head(10)

plt.figure(figsize=(12, 5))

plt.bar(
    top_items["itemid"].astype(str),
    top_items["total_interaction_strength"]
)

plt.title("Top 10 Most Popular Products")

plt.xlabel("Item ID")

plt.ylabel("Total Interaction Strength")

plt.xticks(rotation=45)

plt.show()

## Create a Reusable Recommendation Function

Encapsulate the popularity-based recommendation logic into a reusable function.

This function can later be integrated into the FastAPI application as a fallback recommendation strategy for new users.

In [ ]:
def recommend_popular_items(
    popular_df: pd.DataFrame,
    top_n: int = 10
) -> pd.DataFrame:
    """
    Return the Top-N most popular products.
    """

    return popular_df.head(top_n)

## Test the Recommendation Function

Retrieve the top 10 most popular products using the reusable function.

In [ ]:
recommend_popular_items(
    popular_items,
    top_n=10
)

## Observation

The popularity-based recommendation model provides a simple and effective baseline by recommending products with the highest overall customer engagement.

However, the same recommendations are returned to every user regardless of individual preferences.

To overcome this limitation, the next phase implements **User-Based Collaborative Filtering**, which generates personalized recommendations using similarities between users' interaction histories.

# Phase 2 – User-Based Collaborative Filtering

Popularity-based recommendations provide the same products to every user.

To generate personalized recommendations, we implement **User-Based Collaborative Filtering**.

This approach assumes that users with similar interaction histories are likely to be interested in similar products.

The implementation consists of the following steps:

1. Create a User–Item Interaction Matrix.
2. Compute User Similarity using Cosine Similarity.
3. Identify Similar Users.
4. Generate personalized Top-N product recommendations.

## Create User–Item Interaction Matrix

Transform the interaction dataset into a matrix where:

- Rows represent users.
- Columns represent products.
- Values represent interaction strength.

This matrix forms the foundation of collaborative filtering.

In [ ]:
sample_users = (
    user_item_df["visitorid"]
    .drop_duplicates()
    .sample(
        n=5000,
        random_state=42
    )
)

In [ ]:
sample_df = user_item_df[
    user_item_df["visitorid"].isin(sample_users)
]

In [ ]:
user_item_matrix = sample_df.pivot_table(
    index="visitorid",
    columns="itemid",
    values="interaction_strength",
    fill_value=0
)

In [ ]:
print("Matrix Shape:", user_item_matrix.shape)

user_item_matrix.head()

### Observation

Each row represents a customer.

Each column represents a product.

A higher interaction strength indicates stronger customer interest in the corresponding product.

## Compute User Similarity Matrix

Calculate similarity between users using **Cosine Similarity**.

Cosine Similarity measures how similar two users are based on their interaction vectors.

Users with higher similarity scores are considered behaviorally similar.

In [ ]:
user_similarity = cosine_similarity(
    user_item_matrix
) #nd array obj 

In [ ]:
#nd array obj -> df
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.head()

### Observation

The similarity matrix contains a similarity score for every pair of users.

- A score close to **1** indicates highly similar behavior.
- A score close to **0** indicates very different behavior.

## Create Reusable Function to Find Similar Users

Build a reusable function that returns the most similar users for a given customer.

In [ ]:
def get_similar_users(
    user_id,
    similarity_df,
    top_n=5
):
    """
    Return Top-N most similar users.
    """

    similar_users = (
        similarity_df.loc[user_id] #random user
        .drop(user_id)  #user 101 have similarity 1 with itself so reoved itself
        .sort_values(ascending=False) #highest to lowest similarity 
        .head(top_n)  #if not provided then 5 default 
    )

    return similar_users

## Test Similar User Retrieval

In [ ]:
sample_user = user_item_matrix.index[0]

get_similar_users(
    sample_user,
    user_similarity_df
)

## Generate Personalized Recommendations

The recommendation process consists of:

1. Identify similar users.
2. Retrieve products interacted with by similar users.
3. Exclude products already interacted with by the target user.
4. Rank products by interaction strength.
5. Return the Top-N recommendations.

In [ ]:
def recommend_items(
    user_id,
    user_item_matrix,
    similarity_df,
    top_n=10
):
    """
    Generate Top-N personalized recommendations
    using User-Based Collaborative Filtering.
    """

    if user_id not in user_item_matrix.index:
        return pd.DataFrame() #return empty 

    # Similar users
    similar_users = get_similar_users(
        user_id,
        similarity_df,
        top_n=10
    ).index

    # Items already interacted with
    interacted_items = set(
        user_item_matrix.loc[user_id]
        [
            user_item_matrix.loc[user_id] > 0
        ].index
    )

    recommendations = {}

    for similar_user in similar_users:

        items = user_item_matrix.loc[similar_user]

        for item, score in items.items():

            if score > 0 and item not in interacted_items:

                recommendations[item] = (
                    recommendations.get(item, 0)
                    + score
                )

    recommendation_df = (
        pd.DataFrame(
            recommendations.items(),
            columns=[
                "itemid",
                "recommendation_score"
            ]
        )
        .sort_values(
            "recommendation_score",
            ascending=False
        )
        .head(top_n)
    )

    return recommendation_df

## Test Personalized Recommendations

In [ ]:
recommend_items(
    sample_user,
    user_item_matrix,
    user_similarity_df,
    top_n=10
)

## Validate Recommendations

Generate recommendations for multiple users to verify that the recommendation engine produces personalized outputs.

In [ ]:
sample_users = user_item_matrix.index[:5]

for user in sample_users:

    print("=" * 60)

    print(f"Recommendations for User : {user}")

    print("=" * 60)

    display(                               #display = print df(obj ) in interactive
        recommend_items(
            user,
            user_item_matrix,
            user_similarity_df
        )
    )

### Observation

Unlike popularity-based recommendations, collaborative filtering produces different recommendations for different users.

The recommendations are generated using interaction patterns from behaviorally similar users, making them more personalized and relevant.

> **Note:**  
> This notebook demonstrates **User-Based Collaborative Filtering** for educational purposes and interview explainability. For large-scale production systems, **Item-Based Collaborative Filtering** is generally more scalable and memory-efficient, making it a better choice for deployment in cloud and containerized environments.

# Phase 3 – Save Recommendation Artifacts

To avoid rebuilding the recommendation engine every time the application starts, the generated artifacts are saved for reuse.

These artifacts will later be loaded by the Personalization Engine and FastAPI application, reducing computation time during deployment.

In [ ]:
import joblib
joblib.dump(
    user_item_matrix,
    "../models/user_item_matrix.pkl"
)

print("User-Item Matrix saved successfully.")

## Save User Similarity Matrix

Persist the computed user similarity matrix for reuse during recommendation generation.

In [ ]:
joblib.dump(
    user_similarity_df,
    "../models/user_similarity.pkl"
)

print("User Similarity Matrix saved successfully.")

## Save Popular Products

Store the popularity-based recommendation dataset for use as a fallback recommendation strategy, especially for new users with little or no interaction history.

In [ ]:
popular_items.to_csv(
    "../data/features/popular_items.csv",
    index=False
)

print("Popular Items saved successfully.")

## Verify Saved Artifacts

Confirm that the recommendation artifacts have been successfully written to disk.

In [ ]:
import os

artifacts = [
    "../models/user_item_matrix.pkl",
    "../models/user_similarity.pkl",
    "../data/features/popular_items.csv"
    "../data/features/recommendations.csv
]

for artifact in artifacts:
    if os.path.exists(artifact):
        print(f"✓ {artifact}")
    else:
        print(f"✗ {artifact}")

# Generate Recommendation Dataset

## Objective

Generate Top-N product recommendations for every customer and save them as a reusable dataset.

This dataset will serve as the input for the Feedback System, where historical customer interactions will be compared with generated recommendations to simulate user feedback.

The exported recommendations will also be reused by the Personalization Engine and FastAPI application.

In [ ]:
recommendation_results = []

for user_id in user_item_matrix.index:

    recommendations = recommend_items(
        user_id=user_id,
        user_item_matrix=user_item_matrix,
        similarity_df=user_similarity_df,
        top_n=10
    )

    if recommendations.empty:
        continue

    recommendations["visitorid"] = user_id

    recommendation_results.append(
        recommendations
    )

In [ ]:
recommendations_df = pd.concat(
    recommendation_results,
    ignore_index=True
)

recommendations_df = recommendations_df[
    [
        "visitorid",
        "itemid",
        "recommendation_score"
    ]
]

recommendations_df.head()

In [ ]:
print(f"Shape : {recommendations_df.shape}")

recommendations_df.head()

In [ ]:
recommendations_df.isnull().sum()

## Save Recommendation Dataset

Persist the generated recommendations for downstream modules.

The saved dataset will be consumed by:

- Feedback System
- Personalization Engine
- FastAPI APIs

In [ ]:
recommendations_df.to_csv(
    "../data/features/recommendations.csv",
    index=False
)

print("Recommendations saved successfully.")

## Recommendation Engine Outputs

The following artifacts have been generated:

- **user_item_matrix.pkl** – User–Item interaction matrix.
- **user_similarity.pkl** – User similarity matrix.
- **popular_items.csv** – Globally popular products.
- **recommendations.csv** – Top-N personalized recommendations generated for each customer.

These artifacts will be used by the Feedback System, Personalization Engine, and FastAPI application.

## Conclusion

In this notebook, we successfully developed the recommendation engine for the RecommendIQ project.

The implementation included:

- Loading engineered user-item interaction data.
- Building a popularity-based recommendation baseline.
- Constructing a User–Item interaction matrix.
- Computing user similarity using Cosine Similarity.
- Generating personalized Top-N product recommendations using User-Based Collaborative Filtering.
- Saving reusable recommendation artifacts for deployment.

The generated artifacts provide the foundation for the next stage, where recommendations will be enhanced using customer segmentation in the **Personalization Engine**.